# Instruction Fine-Tuning (SFT) for HR Assistant

This notebook fine-tunes the model on instruction-response pairs for HR Q&A.
Can continue from the non-instruction fine-tuned model or start from base model.

In [37]:
# Install required packages
%pip install -q unsloth transformers datasets accelerate peft trl bitsandbytes

# For Colab, you may need to restart runtime after this cell


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [38]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
import json

# Check GPU availability
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: False


In [39]:
# Load instruction dataset
instruction_data = []
with open('../data/instruction_dataset.jsonl', 'r') as f:
    for line in f:
        instruction_data.append(json.loads(line))

print(f"Loaded {len(instruction_data)} instruction-response pairs")

# Show samples
for i, item in enumerate(instruction_data[:3]):
    print(f"Example {i+1}:")
    print(f"  Instruction: {item['instruction']}")
    print(f"  Response: {item['response'][:150]}...")
    print()

Loaded 120 instruction-response pairs
Example 1:
  Instruction: How often is the company policy reviewed and updated?
  Response: The company policy is reviewed annually and updated as necessary to ensure ongoing compliance with applicable laws and best practices....

Example 2:
  Instruction: Will I be informed if there are major changes to the company policy?
  Response: Yes, employees will be notified of any significant changes to the company policy....

Example 3:
  Instruction: Why does the company review its policy every year?
  Response: The company reviews its policy annually to ensure it remains compliant with current laws and reflects best practices in the industry....



In [40]:
# Format for training - Alpaca style
def format_instruction(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
    }

dataset = Dataset.from_list(instruction_data)
dataset = dataset.map(format_instruction)

# Split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

Map: 100%|██████████| 120/120 [00:00<00:00, 6956.98 examples/s]

Train samples: 108
Eval samples: 12


In [41]:
# Load model - Option 1: Continue from non-instruction FT
# from transformers import AutoModelForCausalLM, AutoTokenizer
# model = AutoModelForCausalLM.from_pretrained(
#     "../models/non_instruction_ft_adapter",
#     torch_dtype=torch.float32,
# )
# tokenizer = AutoTokenizer.from_pretrained("../models/non_instruction_ft_adapter")
# tokenizer.pad_token = tokenizer.eos_token

# Option 2: Start from base model (uncomment if not continuing)
from transformers import AutoModelForCausalLM, AutoTokenizer

max_seq_length = 512

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        "unsloth/Qwen2.5-0.5B",
        torch_dtype=torch.float16,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        "unsloth/Qwen2.5-0.5B",
        torch_dtype=torch.float32,
    )

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-0.5B")
tokenizer.pad_token = tokenizer.eos_token
print("Loaded Hugging Face tokenizer and model for training")

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 120.79it/s]


Loaded Hugging Face tokenizer and model for training


In [42]:
# Apply LoRA/QLoRA
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
print("LoRA adapters attached")

LoRA adapters attached


In [43]:
# Training arguments
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="../models/instruction_ft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    max_steps=200,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_steps=50,
    optim="adamw_torch",
    weight_decay=0.01,
    seed=3407,
    report_to="none",
    dataset_text_field="text",
    max_length=max_seq_length,
)

In [44]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    formatting_func=lambda example: example["text"],
)

Truncating eval dataset: 100%|██████████| 12/12 [00:00<00:00, 7691.27 examples/s]


In [45]:
# Train
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,1.616129,1.325174,1.213600,12151.000000,0.676693
50,0.594908,1.441517,0.760342,23951.000000,0.683605
75,0.323217,1.574112,0.684103,35659.000000,0.666380
100,0.137018,1.790610,0.580539,47520.000000,0.681965
125,0.116172,1.861440,0.550961,59529.000000,0.688018
150,0.092025,2.037698,0.520252,71322.000000,0.677981
175,0.096047,2.100901,0.516903,83100.000000,0.678410
200,0.085406,2.146306,0.512335,94892.000000,0.673215


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [46]:
# Save the model
model.save_pretrained("../models/instruction_ft_adapter")
tokenizer.save_pretrained("../models/instruction_ft_adapter")

# Save a merged/converted checkpoint for inference if desired
print("Saved PEFT adapter and tokenizer to ../models/instruction_ft_adapter")

Saved PEFT adapter and tokenizer to ../models/instruction_ft_adapter


In [48]:
# Test the model after instruction fine-tuning
model.eval()

test_questions = [
    "How can I apply for sick leave?",
    "What is the work from home policy?",
    "How does reimbursement work?",
    "What is the notice period for resignation?",
    "What employee benefits are available?",
    "How is overtime calculated?",
    "What is the onboarding process?",
    "How do I report a compliance concern?",
    "What is the attendance policy?",
    "How are performance reviews conducted?"
]

print("=== INSTRUCTION FT MODEL RESPONSES ===\n")
for q in test_questions:
    prompt = f"### Instruction:\n{q}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part
    response = generated.split("### Response:\n")[-1]
    print(f"Q: {q}")
    print(f"A: {response}")
    print("---")

=== INSTRUCTION FT MODEL RESPONSES ===



AssertionError: Torch not compiled with CUDA enabled

## Compare with Base Model

Run the same 10 questions on the base model (before any fine-tuning) and save responses for comparison report.

In [ ]:
# Load base model for comparison
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    base_model = AutoModelForCausalLM.from_pretrained(
        "unsloth/Qwen2.5-0.5B",
        torch_dtype=torch.float16,
        device_map="auto",
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        "unsloth/Qwen2.5-0.5B",
        torch_dtype=torch.float32,
    )

base_tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-0.5B")
base_tokenizer.pad_token = base_tokenizer.eos_token
base_model.eval()

print("=== BASE MODEL RESPONSES ===\n")
base_responses = {}
for q in test_questions:
    prompt = f"### Instruction:\n{q}\n\n### Response:\n"
    inputs = base_tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = inputs.to("cuda")
    outputs = base_model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    generated = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = generated.split("### Response:\n")[-1]
    base_responses[q] = response
    print(f"Q: {q}")
    print(f"A: {response}")
    print("---")

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 131.47it/s]
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== BASE MODEL RESPONSES ===



Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I apply for sick leave?
A: To apply for sick leave, follow these general steps:

1. Determine if you are eligible for sick leave based on your job or health status.

2. Contact your human resources department to discuss your eligibility for sick leave.

3. Fill out the sick leave application form, including your name, address, and any relevant medical information.

4. Provide proof of your eligibility for sick leave, such as a letter from your employer or a doctor's note.

5. Submit the application to your human resources department.

6. Review the application and any required documentation, and confirm your eligibility for sick leave.

7. Complete any optional forms or paperwork required by your employer.

8. Submit the completed application and any required documentation to your employer's file.

9.
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What is the onboarding process?
A: The onboarding process is the initial phase of an employee's journey at a company. It involves providing initial guidance and support to help the employee become a part of the team and learn about the company's culture, goals, and responsibilities.

The onboarding process typically includes the following steps:

1. Orientation: This is the first step in the onboarding process. It is the initial meeting between the employee and the hiring manager or the human resources department. During this meeting, the employee is introduced to the company's culture, values, and goals, and is given an overview of the company's structure and organizational hierarchy.

2. Training: After the orientation, the employee may undergo a training session or workshop to learn more about the company's products, services
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I report a compliance concern?
A: If you have a concern about your compliance, you should report it to the supervisor. If you don't have access to that person, you can also report it to the Department of Justice.
---


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the attendance policy?
A: Attendance is not required for the final exam.
---
Q: How are performance reviews conducted?
A: Performance reviews are typically conducted by a manager or supervisor, and the process can vary depending on the organization and the nature of the job. However, the general process generally involves the following steps:

1. Define the goals and objectives for the performance review: The manager or supervisor should clearly define the performance goals and objectives for the individual being reviewed.

2. Gather information: The manager or supervisor may gather information from the individual being reviewed, such as their performance history, their performance during the past year, and any other relevant data.

3. Analyze the information: The manager or supervisor will then analyze the information gathered from the individual being reviewed to determine their strengths and weaknesses.

4. Identify areas for improvement: Based on the analysis, the manage

In [ ]:
# Save comparison data for report
comparison_data = {
    "base_model": base_responses,
    "instruction_ft": {}
}

# Re-run instruction FT model responses to save
model.eval()
for q in test_questions:
    prompt = f"### Instruction:\n{q}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = generated.split("### Response:\n")[-1]
    comparison_data["instruction_ft"][q] = response

import json
with open('../../reports/sft_comparison_data.json', 'w') as f:
    json.dump(comparison_data, f, indent=2)

print("Comparison data saved to reports/sft_comparison_data.json")

AssertionError: Torch not compiled with CUDA enabled